# Train LeWM on OGBCubeDR (from a local Hugging Face download)

Trains **LeWM** ([le-wm.github.io](https://le-wm.github.io/)) on the `cube_quadruple_dr_expert` dataset — 5000 domain-randomized `OGBCubeDR-v0` episodes collected with `scripts/data/collect_cube_quadruple_dr_sharded.py` (see `Run.md`).

Unlike `train_from_hf_buckets.ipynb` (which streams a dataset live from HF Buckets with no download), this notebook **downloads the ~27 GB dataset to local disk** first, then trains from the local copy — the right choice on a RunPod pod with a persistent volume, where you want the data to survive pod restarts and don't want every training epoch re-reading over the network.

Steps:
1. Point `STABLEWM_HOME` at persistent storage
2. Authenticate with Hugging Face and download the dataset
3. Authenticate with Weights & Biases
4. Sanity-check the downloaded dataset
5. Launch `scripts/train/lewm.py` with `wandb.enabled=true`

**Prerequisite:** the dataset must already be uploaded to a Hugging Face dataset repo. If it isn't yet, run `upload_to_hf.sh` (repo root, one level above `stable-worldmodel/`) after filling in `HF_TOKEN` — that pushes `datasets/ogbench/cube_quadruple_dr_expert.lance` to `<your-hf-namespace>/ogbench-cube-quadruple-domain-randomized-expert`.

## 1. Repo root and persistent storage

Run this notebook from a `stable-worldmodel` checkout that already has the OGBCubeDR pieces (`stable_worldmodel/envs/ogbench/dr_cube_env.py`, `scripts/data/config/ogb_cube_quadruple_dr.yaml`, etc. — see `Run.md` §1).

On RunPod, `~` lives on the container overlay and is wiped when the pod is recreated. Point **both** `STABLEWM_HOME` and `SPT_CACHE_DIR` at the mounted network volume (typically `/workspace`) **before** downloading anything — the pipeline writes checkpoints through two independent mechanisms:

- `STABLEWM_HOME` — where `scripts/train/lewm.py`'s own `SaveCkptCallback` writes `weights_epoch_N.pt` + `config.json` every epoch (`$STABLEWM_HOME/checkpoints/<output_model_name>/`). This is the artifact `eval_wm.py` consumes.
- `SPT_CACHE_DIR` — where `stable_pretraining.Manager` puts its own Lightning `.ckpt` files (including a `last.ckpt` safety net). It defaults to `~/.cache/stable-pretraining`, which is on the wiped overlay if left unset.

In [ ]:
%cd ../..
!pwd && ls scripts/train/lewm.py

In [ ]:
import os

# Network volume mounted on the pod. Change if yours is mounted elsewhere.
VOLUME_ROOT = '/workspace'

os.environ['STABLEWM_HOME'] = os.path.join(VOLUME_ROOT, 'stable_worldmodel')
os.environ['SPT_CACHE_DIR'] = os.path.join(VOLUME_ROOT, 'cache', 'stable-pretraining')
os.makedirs(os.environ['STABLEWM_HOME'], exist_ok=True)
os.makedirs(os.environ['SPT_CACHE_DIR'], exist_ok=True)

print('STABLEWM_HOME =', os.environ['STABLEWM_HOME'])
print('SPT_CACHE_DIR =', os.environ['SPT_CACHE_DIR'])
!df -h "$VOLUME_ROOT"

## 2. Install dependencies

Skip this cell if the venv was already set up per `Run.md` §3 (`pip install -e '.[train,format]'`). Adds `wandb` and `huggingface_hub` on top.

In [ ]:
%pip install -q -e '.[train,format]' wandb 'huggingface_hub[cli]'

## 3. Hugging Face auth

Needed to download the dataset (required if the repo is private; harmless if it's public but still avoids anonymous-IP rate limits).

In [ ]:
from getpass import getpass

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    os.environ['HF_TOKEN'] = os.environ.get('HF_TOKEN') or getpass('Hugging Face token: ')

## 4. Download the dataset from Hugging Face

Fill in `HF_REPO_ID` below with the namespace `upload_to_hf.sh` pushed to (`<your-hf-username-or-org>/ogbench-cube-quadruple-domain-randomized-expert` unless you changed `REPO_ID` in that script).

This downloads straight into the path `stable_worldmodel.data.load_dataset` expects: `$STABLEWM_HOME/datasets/ogbench/cube_quadruple_dr_expert.lance/`. Re-running is a no-op once the files are present (`huggingface_hub` skips files it already has).

In [ ]:
HF_REPO_ID = '<your-hf-username-or-org>/ogbench-cube-quadruple-domain-randomized-expert'  # ← edit me

DATASET_DIR = os.path.join(
    os.environ['STABLEWM_HOME'], 'datasets', 'ogbench', 'cube_quadruple_dr_expert.lance'
)
os.makedirs(DATASET_DIR, exist_ok=True)
print('Downloading', HF_REPO_ID, '->', DATASET_DIR)

In [ ]:
!hf download "$HF_REPO_ID" --repo-type dataset --local-dir "$DATASET_DIR"

## 5. Weights & Biases auth

In [ ]:
import wandb

try:
    from google.colab import userdata
    wandb_key = userdata.get('WANDB_API_KEY')
except Exception:
    wandb_key = os.environ.get('WANDB_API_KEY') or getpass('Weights & Biases API key: ')

wandb.login(key=wandb_key)

`scripts/train/config/launcher/local.yaml` defaults `wandb.config.entity`/`project` to `stable-wm`, which is almost certainly not your account. Set your own below — passed as Hydra overrides in the training cell.

In [ ]:
WANDB_ENTITY = '<your-wandb-entity>'   # ← edit me
WANDB_PROJECT = 'ogbcubedr-lewm'       # ← edit me if you want a different project name

## 6. Sanity-check the dataset and GPU

Confirms the download landed correctly (5000 episodes x 401 steps expected, per `Run.md` §5.6) and that a GPU is visible — `lewm.yaml` hardcodes `accelerator: gpu`.

In [ ]:
import lance

ds = lance.dataset(DATASET_DIR)
ep = ds.to_table(columns=['episode_idx']).column('episode_idx').to_numpy()
print(f'rows: {ds.count_rows():,}')
print(f'episodes: {ep.max() - ep.min() + 1}')

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU visible — lewm.yaml requires accelerator: gpu'
print(torch.cuda.get_device_name(0))

## 7. Train

Optional smoke run first — a couple of epochs on a small batch, just to catch a config/schema break before committing GPU-hours to the full run.

In [ ]:
!python scripts/train/lewm.py data=ogb_cube_quadruple_dr \
    output_model_name=lewm_q4_dr_smoke \
    trainer.max_epochs=1 \
    loader.batch_size=8 loader.num_workers=0 \
    wandb.enabled=false \
    hydra.run.dir=/tmp/lewm_smoke

Full run, logging to W&B:

In [ ]:
!python scripts/train/lewm.py data=ogb_cube_quadruple_dr \
    output_model_name=lewm_q4_dr \
    wandb.enabled=true \
    wandb.config.entity="$WANDB_ENTITY" \
    wandb.config.project="$WANDB_PROJECT"

Checkpoints land in two places, both now on the network volume:

- `$STABLEWM_HOME/checkpoints/lewm_q4_dr/weights_epoch_N.pt` + `config.json` — written every epoch, this is what `scripts/plan/eval_wm.py` loads (`policy=lewm_q4_dr/weights_epoch_N.pt`).
- `$SPT_CACHE_DIR/runs/<date>/<time>/<uuid>/checkpoints/` — Lightning's own `.ckpt` files (including `last.ckpt`), useful only for resuming a Lightning `Trainer` session directly; not consumed by `eval_wm.py`.

See `Run.md` §6 for the columns to watch during training (`pred_loss` falling, `sigreg_loss` bounded) and §7 for evaluation.